# MedBoard — Train U-Net Segmentation Model

**Run this notebook in Google Colab to train the segmentation model.**

What this notebook does:
1. Set up the environment (mount Drive, clone repo, copy data)
2. Load BRISC 2025 segmentation dataset
3. Train U-Net with Dice+BCE loss and early stopping
4. Save best weights to Google Drive
5. Plot training curves

**Expected training time:** ~1.5–2 hours on a free Colab T4 GPU

## Cell 1 — Environment Setup (Run Every Session)

In [ ]:
from google.colab import drive
import shutil, os, sys

# Mount Google Drive
drive.mount('/content/drive')

# Clone or update the MedBoard repo from GitHub
if not os.path.exists('/content/MedBoard'):
    !git clone https://github.com/shivanshu0055/Medical-Image.git /content/MedBoard
else:
    !git -C /content/MedBoard pull

# Copy dataset from Drive to local SSD (faster I/O during training)
if not os.path.exists('/content/data'):
    print('Copying dataset to local SSD (~30 sec)...')
    shutil.copytree('/content/drive/MyDrive/medboard/data/brisc2025', '/content/data')
    print('Done.')
else:
    print('Dataset already on local SSD.')

# Install dependencies
!pip install -q -r /content/MedBoard/requirements_colab.txt

# Set Python path so we can import from MedBoard
sys.path.insert(0, '/content/MedBoard')

# Define all paths
DATA_ROOT        = '/content/data'
WEIGHTS_DIR      = '/content/drive/MyDrive/medboard/weights'
CHECKPOINT_PATH  = f'{WEIGHTS_DIR}/unet_best.pth'
FINAL_PATH       = f'{WEIGHTS_DIR}/unet_brisc.pth'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

print(f'\nSetup complete!')
print(f'  Data    → {DATA_ROOT}')
print(f'  Weights → {WEIGHTS_DIR}')

## Cell 2 — Verify GPU

In [ ]:
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('WARNING: No GPU detected! Training will be very slow on CPU.')
    print('Go to Runtime → Change runtime type → T4 GPU')

## Cell 3 — Load Dataset

In [ ]:
from modules.seg_dataset import get_seg_dataloaders

train_loader, val_loader, test_loader = get_seg_dataloaders(
    train_images_dir = f'{DATA_ROOT}/segmentation_task/train/images',
    train_masks_dir  = f'{DATA_ROOT}/segmentation_task/train/masks',
    test_images_dir  = f'{DATA_ROOT}/segmentation_task/test/images',
    test_masks_dir   = f'{DATA_ROOT}/segmentation_task/test/masks',
    batch_size  = 8,      # safe for Colab T4 (16GB VRAM)
    num_workers = 2,      # Colab usually has 2 CPU cores
    val_split   = 0.1,
    config_path = '/content/MedBoard/configs/config.yaml',
)

## Cell 4 — Build Model

In [ ]:
import yaml
from modules.unet import build_unet, count_parameters

with open('/content/MedBoard/configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

model = build_unet(cfg['segmentation'])
print(f'U-Net parameters: {count_parameters(model):,}  ({count_parameters(model)/1e6:.2f}M)')

# Quick shape check
dummy = torch.randn(1, 3, 256, 256)
out   = model(dummy)
print(f'Input:  {tuple(dummy.shape)}')
print(f'Output: {tuple(out.shape)}  (expected: (1, 1, 256, 256))')

## Cell 5 — Train

In [ ]:
from modules.seg_trainer import train_segmentation

history = train_segmentation(
    model           = model,
    train_loader    = train_loader,
    val_loader      = val_loader,
    epochs          = cfg['segmentation']['epochs'],                   # 40
    learning_rate   = cfg['segmentation']['learning_rate'],            # 0.0001
    patience        = cfg['segmentation']['early_stopping_patience'],  # 7
    checkpoint_path = CHECKPOINT_PATH,
    device_str      = 'cuda',
)

## Cell 6 — Save Final Weights to Drive

In [ ]:
import shutil

# Copy best checkpoint to final weights path
shutil.copy(CHECKPOINT_PATH, FINAL_PATH)
print(f'Final weights saved to: {FINAL_PATH}')
print(f'Best val dice: {max(history["val_dice"]):.4f}')

## Cell 7 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_ran = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.plot(epochs_ran, history['train_loss'], label='Train Loss', color='royalblue')
ax1.plot(epochs_ran, history['val_loss'],   label='Val Loss',   color='tomato')
ax1.set_title('Loss per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Dice + BCE Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Dice score curve
ax2.plot(epochs_ran, history['train_dice'], label='Train Dice', color='royalblue')
ax2.plot(epochs_ran, history['val_dice'],   label='Val Dice',   color='tomato')
ax2.axhline(0.75, color='green', linestyle='--', alpha=0.5, label='Good (0.75)')
ax2.axhline(0.85, color='gold',  linestyle='--', alpha=0.5, label='Excellent (0.85)')
ax2.set_title('Dice Score per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Dice Score')
ax2.set_ylim([0, 1])
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('U-Net Segmentation Training — MedBoard', fontsize=14, fontweight='bold')
plt.tight_layout()

# Save plot to Drive
plot_path = f'{WEIGHTS_DIR}/seg_training_curves.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved to {plot_path}')

## Cell 8 — Evaluate on Test Set

In [ ]:
from modules.seg_trainer import validate_one_epoch, DiceBCELoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = DiceBCELoss()

test_loss, test_dice = validate_one_epoch(model, test_loader, criterion, device)

print('=== Test Set Results ===')
print(f'Test Loss:  {test_loss:.4f}')
print(f'Test Dice:  {test_dice:.4f}')
print()
if test_dice >= 0.85:
    print('Excellent result!')
elif test_dice >= 0.75:
    print('Good result — model is usable.')
else:
    print('Below target. Consider more epochs or tuning hyperparameters.')